# Visualization

This notebook serves to visualize the results of the models.

In [1]:
import os
import shutil
import h5py
import numpy as np
import sys
import pandas as pd
import json
import matplotlib.pyplot as plt
%matplotlib inline
import importlib

sys.path.append("..")
sys.path.append("../code")
sys.path.append(os.path.join("..", 'models','Pointnet_Pointnet2_pytorch', 'models'))

from dataset import PCExtrusionSegmentationDataset
from models.DeepCAD.cadlib.visualize import vec2CADsolid
from OCC.Core.BRepCheck import BRepCheck_Analyzer
from OCC.Extend.DataExchange import write_step_file
from OCC.Core.STEPControl import STEPControl_Reader
from OCC.Core.StlAPI import StlAPI_Writer
from OCC.Core.BRepMesh import BRepMesh_IncrementalMesh
from models.DeepCAD.cadlib.extrude import CADSequence
from models.DeepCAD.cadlib.visualize import create_CAD
from models.DeepCAD.cadlib.visualize import CADsolid2pc
from models.DeepCAD.utils.pc_utils import write_ply
import open3d as o3d
from metrics import ClassificationRunningScore
import torch

## Extrusion Segmentation

### Visualization

In [2]:
def get_trained_segmentation_pn2(model_path):
   
    model_name = 'pointnet2_sem_seg_msg'
    model = importlib.import_module(model_name)
    num_classes = 10
    classifier = model.get_model(num_classes)
    
    trained_model = torch.load(model_path, map_location=torch.device('cpu'), weights_only=True)
    state_dict = trained_model['model_state_dict']
    classifier.load_state_dict(state_dict)
    config = trained_model['config']

    return classifier, config
    

In [3]:
import open3d as o3d
import matplotlib.pyplot as plt

def visualize_labeled_pc(points, labels):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)

    colors = plt.cm.tab10(labels / labels.max())[:, :3] 
    pcd.colors = o3d.utility.Vector3dVector(colors)

    o3d.visualization.draw_geometries([pcd])

In [4]:
def infer_segmentation_pn2(model, pc, show=True):
    pc = pc.unsqueeze(0)
    pc = pc.transpose(2, 1)
    pred_logits, _ = model(pc)
    pred_logits = pred_logits.data.view(-1, 10)
    pred = pred_logits.max(1)[1]
    return pred

In [27]:
run_name = "partseg_overfitted_on_train_0"
model_path = os.path.join("..", "models", "trained_models", run_name, "best.pth")
classifier, config = get_trained_segmentation_pn2(model_path)
config

{'learning_rate': 0.001,
 'batch_size': 2,
 'max_epochs': 50,
 'optimizer': 'Adam',
 'model_type': 'pointnet2_sem_seg_msg',
 'save_interval': 20,
 'early_stopping': 20,
 'start_time': '2025-06-26_17-09-47',
 'lr_type': 'step',
 'gpu': True,
 'final_epoch': 49,
 'training_time_min': 5.67}

In [6]:
train_dataset = PCExtrusionSegmentationDataset("../data", 'train', use_normals=False, verbose=False)
val_dataset = PCExtrusionSegmentationDataset("../data", 'validation', use_normals=False, verbose=False)
test_dataset = PCExtrusionSegmentationDataset("../data", 'test', use_normals=False, verbose=False)

In [36]:
index = 0
data = train_dataset[index]
pc = data['pc']
label = data['label']

In [37]:
visualize_labeled_pc(pc, label)
pred_labels = infer_segmentation_pn2(classifier, pc)
visualize_labeled_pc(pc, pred_labels)

[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display


In [22]:
for i in range(120000, 120100):
    data = train_dataset[i]
    pc = data['pc']
    label = data['label']
    visualize_labeled_pc(pc, label)
    pred_labels = infer_segmentation_pn2(classifier, pc)
    visualize_labeled_pc(pc, pred_labels)

[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARN

KeyboardInterrupt: 

In [32]:
pred_labels.shape

torch.Size([2048])

In [31]:
import torch
import numpy as np
from plyfile import PlyData, PlyElement
import matplotlib.pyplot as plt

index = 0
data = train_dataset[index]
points = data['pc']
labels = data['label']
print(labels.shape)

labels = pred_labels

points_np = points.numpy()
labels_np = labels.numpy()

# Convert labels to RGB using colormap
cmap = plt.get_cmap("tab10")  # 10 distinct colors
colors = (np.array([cmap(label)[:3] for label in labels_np]) * 255).astype(np.uint8)

# Combine into structured array for PLY
vertex_data = np.array(
    [(points_np[i, 0], points_np[i, 1], points_np[i, 2], colors[i, 0], colors[i, 1], colors[i, 2])
     for i in range(points_np.shape[0])],
    dtype=[("x", "f4"), ("y", "f4"), ("z", "f4"),
           ("red", "u1"), ("green", "u1"), ("blue", "u1")]
)

# Save to PLY
el = PlyElement.describe(vertex_data, "vertex")
PlyData([el], text=True).write("examples/pred_labeled_pointcloud.ply")
print("Saved labeled_pointcloud.ply")


torch.Size([2048])
Saved labeled_pointcloud.ply


In [38]:
import open3d as o3d
import numpy as np
import torch

points = pc
labels = pred_labels

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points.numpy())

# Store labels as a scalar color channel (temporary workaround using red)
colors = np.zeros((points.shape[0], 3))
color_map = np.array([
    [0.894, 0.102, 0.110],  # class 0
    [0.216, 0.494, 0.722],  # class 1
    [0.302, 0.686, 0.290],  # class 2
    [0.596, 0.306, 0.639],  # class 3
    [1.000, 0.498, 0.0],    # class 4
    [1.000, 1.000, 0.2],    # class 5
    [0.651, 0.337, 0.157],  # class 6
    [0.969, 0.506, 0.749],  # class 7
    [0.6,   0.6,   0.6],    # class 8
    [0.1,   0.1,   0.1],    # class 9
])
colors = color_map[labels.numpy()]
pcd.colors = o3d.utility.Vector3dVector(colors)

o3d.io.write_point_cloud("examples/pred_labeled_cloud.ply", pcd)


True

### Test Metrics

In [9]:
test_metrics_1 = np.load(os.path.join("..", "models", "trained_models", run_name, "test_metrics.npz"), allow_pickle=True)
test_metrics_2 = pd.read_csv(os.path.join("..", "models", "trained_models", run_name, "test_metrics.csv"))

In [11]:
test_metrics_2

,mIoU,acc,mean_acc,loss
0,0.231052,0.821965,0.298237,0.696476


In [12]:
test_metrics_1['class_iou'], test_metrics_1['class_acc']

(array([[0.88004574, 0.39655013, 0.23347675, 0.17249301, 0.14397624,
         0.10523883, 0.09056356, 0.0794565 , 0.11344418, 0.09527454]]),
 array([[0.95592844, 0.55201697, 0.36310666, 0.25237258, 0.22397793,
         0.15413569, 0.1289525 , 0.10086377, 0.1409144 , 0.11010391]]))